In [2]:
#!/usr/bin/env python3
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

from lerobot_so_arm.config import get_path
from lerobot_so_arm.utils.kinematics import RobotKinematics
from lerobot_so_arm.utils.visualization import compute_joint_positions_in_world_coordinates, add_base_coordinate_system

vjepa_root = get_path('vjepa_root')
sys.path.append(vjepa_root)

# Robot joint limits configuration based on calibration data
ROBOT_CONFIG = {
    "joint_limits": {
        "shoulder_pan": {"min": -110.0, "max": 120.0, "default": 0.0},
        "shoulder_lift": {"min": -103.0, "max": 102.0, "default": 0.0},
        "elbow_flex": {"min": -103.0, "max": 88.0, "default": 0.0},
        "wrist_flex": {"min": -98.0, "max": 99.0, "default": 0.0},
        "wrist_roll": {"min": -172.0, "max": 166.0, "default": 0.0},
        "gripper": {"min": 0.0, "max": 100.0, "default": 0.0}
    },
    "preset_positions": {
        "home": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "extended": [0.0, 90.0, 0.0, 0.0, 0.0, 0.0],
        "folded": [0.0, 90.0, 90.0, 0.0, 0.0, 0.0]
    }
}

# Install plotly if not already installed
try:
    import plotly
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print(f"Plotly version: {plotly.__version__}")
except ImportError:
    print("Installing plotly...")
    %pip install plotly --quiet
    import plotly
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print(f"Plotly installed, version: {plotly.__version__}")

# Initialize robot kinematics for SO100/SO101
calibration_name = "so_new_calibration"
#calibration_name = "so_old_calibration"
robot_kinematics = RobotKinematics(calibration_name)
print(f"Robot kinematics initialized for: {calibration_name}")


ModuleNotFoundError: No module named 'lerobot_so_arm'

In [ ]:
# Function to add dotted coordinate axes with labels (same as in interactive visualizer)
def add_dotted_coordinate_axes(fig):
    """Add dotted coordinate axes with colored labels, no legend entries"""
    
    # X-axis: Vertical (up) - red dotted line
    fig.add_trace(go.Scatter3d(
        x=[0, 0],
        y=[0, 0],
        z=[-0.1, 0.5],
        mode='lines',
        line=dict(color='red', width=3, dash='dot'),
        name='X-axis',
        showlegend=False,
        hoverinfo='skip'
    ))
    # X-axis arrow
    fig.add_trace(go.Scatter3d(
        x=[-0.025, 0, 0.025],
        y=[0, 0, 0],
        z=[0.45, 0.5, 0.45],
        mode='lines',
        line=dict(color='red', width=3),
        name='X-arrow',
        showlegend=False,
        hoverinfo='skip'
    ))
    # X-axis label (red)
    fig.add_trace(go.Scatter3d(
        x=[0],
        y=[0],
        z=[0.52],
        mode='text',
        text=['X'],
        textposition='middle center',
        textfont=dict(color='red', size=16),
        name='X-label',
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Y-axis: Forward (arm extension) - green dotted line
    fig.add_trace(go.Scatter3d(
        x=[0, 0],
        y=[-0.5, 0.5],
        z=[0, 0],
        mode='lines',
        line=dict(color='green', width=3, dash='dot'),
        name='Y-axis',
        showlegend=False,
        hoverinfo='skip'
    ))
    # Y-axis arrow
    fig.add_trace(go.Scatter3d(
        x=[0, 0, 0],
        y=[0.45, 0.5, 0.45],
        z=[-0.025, 0, 0.025],
        mode='lines',
        line=dict(color='green', width=3),
        name='Y-arrow',
        showlegend=False,
        hoverinfo='skip'
    ))
    # Y-axis label (green)
    fig.add_trace(go.Scatter3d(
        x=[0],
        y=[0.52],
        z=[0],
        mode='text',
        text=['Y'],
        textposition='middle center',
        textfont=dict(color='green', size=16),
        name='Y-label',
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Z-axis: Left (perpendicular) - blue dotted line
    fig.add_trace(go.Scatter3d(
        x=[-0.5, 0.5],
        y=[0, 0],
        z=[0, 0],
        mode='lines',
        line=dict(color='blue', width=3, dash='dot'),
        name='Z-axis',
        showlegend=False,
        hoverinfo='skip'
    ))
    # Z-axis arrow
    fig.add_trace(go.Scatter3d(
        x=[0.45, 0.5, 0.45],
        y=[0, 0, 0],
        z=[-0.025, 0, 0.025],
        mode='lines',
        line=dict(color='blue', width=3),
        name='Z-arrow',
        showlegend=False,
        hoverinfo='skip'
    ))
    # Z-axis label (blue)
    fig.add_trace(go.Scatter3d(
        x=[0.52],
        y=[0],
        z=[0],
        mode='text',
        text=['Z'],
        textposition='middle center',
        textfont=dict(color='blue', size=16),
        name='Z-label',
        showlegend=False,
        hoverinfo='skip'
    ))


In [ ]:
# Note: compute_robot_frames function is now imported from lerobot_so_arm.utils.visualization

def add_coordinate_axes(fig, position, orientation, scale=0.03, name="", base_color=None):
    """
    Add coordinate axes to the 3D plot at the specified position and orientation
    
    Args:
        fig: Plotly figure
        position: [x, y, z] position of the coordinate system origin
        orientation: 3x3 rotation matrix defining the orientation
        scale: Size of the coordinate axes
        name: Name prefix for the axes
        base_color: Optional color for the coordinate system origin
        
    Returns:
        Updated figure
    """
    # Extract position components
    x, y, z = position
    
    # Extract orientation vectors (columns of the rotation matrix)
    x_axis = orientation[:, 0] * scale
    y_axis = orientation[:, 1] * scale
    z_axis = orientation[:, 2] * scale
    
    # Standard axis colors with transparency
    axis_colors = {
        'x': 'rgba(255, 0, 0, 0.7)',  # Red with transparency
        'y': 'rgba(0, 255, 0, 0.7)',  # Green with transparency
        'z': 'rgba(0, 0, 255, 0.7)'   # Blue with transparency
    }
    
    # If base_color is provided, add a small marker at the origin
    if base_color:
        fig.add_trace(go.Scatter3d(
            x=[x],
            y=[y],
            z=[z],
            mode='markers',
            marker=dict(color=base_color, size=4, opacity=0.8),
            name=f'{name}',
            hoverinfo='name',
            showlegend=False
        ))
    
    # Add X-axis (red)
    fig.add_trace(go.Scatter3d(
        x=[x, x + x_axis[0]],
        y=[y, y + x_axis[1]],
        z=[z, z + x_axis[2]],
        mode='lines',
        line=dict(color=axis_colors['x'], width=2),
        name=f'{name} X-axis',
        hoverinfo='name',
        showlegend=False
    ))
    
    # Add Y-axis (green)
    fig.add_trace(go.Scatter3d(
        x=[x, x + y_axis[0]],
        y=[y, y + y_axis[1]],
        z=[z, z + y_axis[2]],
        mode='lines',
        line=dict(color=axis_colors['y'], width=2),
        name=f'{name} Y-axis',
        hoverinfo='name',
        showlegend=False
    ))
    
    # Add Z-axis (blue)
    fig.add_trace(go.Scatter3d(
        x=[x, x + z_axis[0]],
        y=[y, y + z_axis[1]],
        z=[z, z + z_axis[2]],
        mode='lines',
        line=dict(color=axis_colors['z'], width=2),
        name=f'{name} Z-axis',
        hoverinfo='name',
        showlegend=False
    ))
    
    return fig

def visualize_robot(joint_positions, fig=None):
    """
    Create a 3D visualization of the robot based on joint positions
    
    Args:
        joint_positions: Array of 6 joint positions [shoulder_pan, shoulder_lift, elbow_flex, wrist_flex, wrist_roll, gripper]
        fig: Optional existing figure to update (for interactive updates)
        
    Returns:
        Plotly figure with robot visualization
    """
    # Compute frame positions using the function from visualization.py
    frames = compute_joint_positions_in_world_coordinates(joint_positions, robot_kinematics)
    
    # Create figure if not provided
    if fig is None:
        fig = go.Figure()
    
    # Always update the title with current joint positions
    fig.update_layout(
        title=f"Robot Kinematic Chain - Joint Positions: {joint_positions.round(1)}",
    )
    
    # Only set these layout properties if it's a new figure
    if fig.data == () or not fig.data:  # If this is a new figure or all traces were cleared
        fig.update_layout(
            scene=dict(
                xaxis_title='X (m)',
                yaxis_title='Y (m)',
                zaxis_title='Z (m)',
                # Set fixed axis ranges for stability
                xaxis=dict(range=[-0.6, 0.6], autorange=False),
                yaxis=dict(range=[-0.6, 0.6], autorange=False),
                zaxis=dict(range=[-0.1, 1.0], autorange=False),
                # Use cube aspect ratio for better proportions
                aspectmode='cube',
                # Add camera settings for a stable view
                camera=dict(
                    eye=dict(x=1.5, y=1.5, z=0.8),
                    up=dict(x=0, y=0, z=1)
                ),
                # Add grid for better spatial reference
                xaxis_showgrid=True,
                yaxis_showgrid=True,
                zaxis_showgrid=True,
                # Make background white for cleaner look
                bgcolor='white'
            ),
            width=800,
            height=700,
            margin=dict(l=0, r=0, b=0, t=40),
            legend=dict(
                # Move legend outside of the figure to the right
                yanchor="top",
                y=1,
                xanchor="left",
                x=1.02,
                # Improve legend layout
                bgcolor="rgba(255, 255, 255, 0.5)",
                bordercolor="rgba(0, 0, 0, 0.5)",
                borderwidth=1
            ),
            autosize=True,
            template="plotly_white"
        )
    
    # Define frame colors for visualization
    frame_colors = {
        "base": "black",
        "shoulder": "blue",
        "humerus": "green",
        "forearm": "orange",
        "wrist": "purple",
        "gripper": "red",
        "gripper_tip": "pink"
    }
    
    # Extract positions for all frames
    positions = {frame: data['position'] for frame, data in frames.items()}
    
    # Define the kinematic chain connections
    connections = [
        ("base", "shoulder"),
        ("shoulder", "humerus"),
        ("humerus", "forearm"),
        ("forearm", "wrist"),
        ("wrist", "gripper"),
        ("gripper", "gripper_tip")
    ]
    
    # Add lines connecting the frames first (so they appear behind the markers)
    for start, end in connections:
        start_pos = positions[start]
        end_pos = positions[end]
        
        # Use the same color as the origin joint with some transparency
        segment_color = frame_colors[start]
        # Convert to rgba to add transparency
        if isinstance(segment_color, str) and segment_color.startswith('#'):
            # Convert hex to rgba
            r = int(segment_color[1:3], 16)
            g = int(segment_color[3:5], 16)
            b = int(segment_color[5:7], 16)
            segment_color = f'rgba({r}, {g}, {b}, 0.8)'
        
        fig.add_trace(go.Scatter3d(
            x=[start_pos[0], end_pos[0]],
            y=[start_pos[1], end_pos[1]],
            z=[start_pos[2], end_pos[2]],
            mode='lines',
            line=dict(color=segment_color, width=8),
            name=f"{start}-{end}",
            hoverinfo='name',
            showlegend=False  # Hide segments from legend to keep it clean
        ))
    
    # Add markers for each frame
    for frame, position in positions.items():
        # Only add text for key frames to reduce clutter
        mode = 'markers+text' if frame in ['base', 'gripper', 'gripper_tip'] else 'markers'
        
        fig.add_trace(go.Scatter3d(
            x=[position[0]],
            y=[position[1]],
            z=[position[2]],
            mode=mode,
            marker=dict(
                size=8, 
                color=frame_colors[frame],
                symbol='circle',
                line=dict(color='black', width=1),  # Add black outline
                opacity=0.9
            ),
            text=[frame],
            textposition="top center",
            name=frame,
            hoverinfo='name+text',
            showlegend=False  # Hide from legend
        ))
    
    # Add coordinate axes for each frame
    for frame, data in frames.items():
        # Use a smaller scale for coordinate axes to make them less intrusive
        add_coordinate_axes(
            fig, 
            data['position'], 
            data['orientation'],
            scale=0.025,
            name=frame,
            base_color=frame_colors[frame]  # Use frame color for coordinate system
        )
    
    return fig


In [ ]:
# Create interactive sliders for joint positions
def create_joint_sliders():
    """Create sliders for controlling joint positions using the ROBOT_CONFIG"""
    
    # Create a dictionary to hold all sliders
    sliders = {}
    
    # Create sliders for each joint using the configuration
    for joint_name, limits in ROBOT_CONFIG["joint_limits"].items():
        sliders[joint_name] = widgets.FloatSlider(
            value=limits["default"],
            min=limits["min"],
            max=limits["max"],
            step=1.0,
            description=f"{joint_name.replace('_', ' ').title()}:",
            disabled=False,
            continuous_update=True,
            orientation='horizontal',
            readout=True,
            readout_format='.1f',
            layout=widgets.Layout(width='80%')
        )
    
    return sliders

# Create sliders
joint_sliders = create_joint_sliders()

# Create output widget for the visualization
output = widgets.Output()

# Create a single figure that we'll update
from plotly.graph_objs import FigureWidget
fig_widget = FigureWidget()

# Initialize the figure with proper layout
fig_widget.update_layout(
    uirevision='constant',  # Prevent camera from auto-adjusting during updates
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        # Set fixed axis ranges for stability
        xaxis=dict(range=[-0.6, 0.6], autorange=False),
        yaxis=dict(range=[-0.6, 0.6], autorange=False),
        zaxis=dict(range=[-0.1, 1.0], autorange=False),
        # Use cube aspect ratio for better proportions
        aspectmode='cube',
        # Add camera settings for a stable view
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.8),
            up=dict(x=0, y=0, z=1)
        ),
        # Add grid for better spatial reference
        xaxis_showgrid=True,
        yaxis_showgrid=True,
        zaxis_showgrid=True,
        # Make background white for cleaner look
        bgcolor='white'
    ),
    width=800,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(
        # Move legend outside of the figure to the right
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        # Improve legend layout
        bgcolor="rgba(255, 255, 255, 0.5)",
        bordercolor="rgba(0, 0, 0, 0.5)",
        borderwidth=1
    ),
    # Make sure the plot fits in the available space
    autosize=True,
    # Use a clean template
    template="plotly_white"
)

# Note: Using add_base_coordinate_system from lerobot_so_arm.utils.visualization

# Function to update the visualization based on slider values
def update_visualization(change=None):
    # Get current joint positions from sliders (in degrees)
    joint_positions_degrees = np.array([
        joint_sliders['shoulder_pan'].value,
        joint_sliders['shoulder_lift'].value,
        joint_sliders['elbow_flex'].value,
        joint_sliders['wrist_flex'].value,
        joint_sliders['wrist_roll'].value,
        joint_sliders['gripper'].value
    ])
    
    # Use degrees directly for the kinematics calculations
    joint_positions = joint_positions_degrees.copy()
    
    # Update the existing figure
    # First, clear all existing traces
    fig_widget.data = []
    
    # Add dotted coordinate axes with labels (no legend entries)
    add_dotted_coordinate_axes(fig_widget)
    
    # Then update the figure with robot data
    visualize_robot(joint_positions, fig=fig_widget)
    
    # Display the figure only once on the first call
    if not hasattr(update_visualization, 'initialized'):
        with output:
            output.clear_output(wait=True)
            display(fig_widget)
            update_visualization.initialized = True

# Register callbacks for all sliders
for slider in joint_sliders.values():
    slider.observe(update_visualization, names='value')

# Create a button for resetting to default position
reset_button = widgets.Button(
    description='Reset Position',
    disabled=False,
    button_style='',
    tooltip='Reset joint positions to default values',
    icon='refresh'
)

# Define reset function
def reset_position(b):
    # Reset to default values from configuration
    for joint_name, limits in ROBOT_CONFIG["joint_limits"].items():
        joint_sliders[joint_name].value = limits["default"]
    
    # Update visualization
    update_visualization()

# Register callback for reset button
reset_button.on_click(reset_position)

# Create preset position buttons
home_button = widgets.Button(
    description='Home Position',
    disabled=False,
    button_style='',
    tooltip='Move to home position',
    icon='home'
)

extended_button = widgets.Button(
    description='Extended Position',
    disabled=False,
    button_style='',
    tooltip='Move to extended position',
    icon='arrow-right'
)

folded_button = widgets.Button(
    description='Folded Position',
    disabled=False,
    button_style='',
    tooltip='Move to folded position',
    icon='compress'
)

# Define preset functions
def set_preset_position(preset_name):
    """Set a preset position from the configuration"""
    if preset_name in ROBOT_CONFIG["preset_positions"]:
        positions = ROBOT_CONFIG["preset_positions"][preset_name]
        joint_names = list(ROBOT_CONFIG["joint_limits"].keys())
        
        for i, joint_name in enumerate(joint_names):
            if i < len(positions):
                # Make sure the position is within limits
                min_val = ROBOT_CONFIG["joint_limits"][joint_name]["min"]
                max_val = ROBOT_CONFIG["joint_limits"][joint_name]["max"]
                value = max(min_val, min(max_val, positions[i]))
                joint_sliders[joint_name].value = value

def home_position(b):
    set_preset_position("home")

def extended_position(b):
    set_preset_position("extended")

# Add a folded position button
def folded_position(b):
    set_preset_position("folded")

# Register callbacks for preset buttons
home_button.on_click(home_position)
extended_button.on_click(extended_position)
folded_button.on_click(folded_position)

# Create a horizontal box for buttons
button_box = widgets.HBox([reset_button, home_button, extended_button, folded_button])

# Display sliders and buttons
controls = widgets.VBox([
    widgets.HTML("<h3>Joint Position Controls</h3>"),
    joint_sliders['shoulder_pan'],
    joint_sliders['shoulder_lift'],
    joint_sliders['elbow_flex'],
    joint_sliders['wrist_flex'],
    joint_sliders['wrist_roll'],
    joint_sliders['gripper'],
    button_box
])

# Set layout for controls and output
controls.layout.width = '100%'  # Full width for controls
output.layout.width = '100%'    # Full width for output
output.layout.min_height = '700px'  # Ensure output has enough height

# Create a vertical layout with controls on top and visualization below
display(widgets.VBox([
    controls,
    output
]))

# Initial visualization - make sure the figure is displayed
with output:
    output.clear_output(wait=True)
    display(fig_widget)
    update_visualization.initialized = True

# Update the visualization with initial values
update_visualization()
